# Stem-Modell trainieren (Google Colab)

Finetuning eines BS-RoFormer-Modells für den airdox_SMART_Editor, mit Export in das Format,
das `AudioSeparatorSeparator` in der Desktop-App lädt.

---

## Vorab drei Dinge, die du wissen musst

**1. Wir trainieren NICHT von Null.** BS-RoFormer from scratch auf MUSDB18-HQ kostet laut
Literatur mehrere A100-GPUs über Tage bis Wochen. Colab gibt dir eine T4/L4 mit Session-Limit
(üblich: ~4 h in der Gratis-Stufe, danach Abbruch). From-scratch-Training endet dort garantiert
in einem unbrauchbaren Modell. Wir machen **Finetuning** auf einem vortrainierten Checkpoint:
das ist in Stunden sinnvoll machbar und liefert ein Modell, das tatsächlich trennt.

**2. Die Lizenzlage ist der Knackpunkt, nicht die Technik.**

| Bestandteil | Lizenz | Kommerziell nutzbar? |
| --- | --- | --- |
| MSST-Trainingscode (ZFTurbo) | MIT | ja |
| MUSDB18 / MUSDB18-HQ (Daten) | *educational purposes only*, Teile CC BY-NC-SA | **nein** |
| Vortrainierte ZFTurbo-Gewichte | keine explizite Lizenz | **ungeklärt** |
| Dein Finetune-Ergebnis | erbt die Einschränkungen von Daten + Startgewichten | **nein** |

Ein auf MUSDB18 finetuntes Modell ist ein **Forschungs- und Entwicklungsartefakt**. Für ein
kommerzielles Release brauchst du entweder eigene/lizenzierte Multitracks oder die schriftliche
Freigabe der Rechteinhaber. Das ist keine Formalie — MUSDB18 sagt das wörtlich.

**3. Was am Ende herauskommt.** Ein `.ckpt` + `.yaml`, die du lokal nach `models/bsroformer/`
legst. Die App nutzt sie über `modelFilename`. Das Stem Isolation Gate entscheidet dann, ob das
Ergebnis `RELEASE_READY` ist — nicht dieses Notebook und nicht die Trainings-Loss.


## 0. GPU prüfen

Ohne GPU nicht weitermachen: `Laufzeit → Laufzeittyp ändern → GPU`.


In [ ]:
import subprocess, sys
out = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                     capture_output=True, text=True)
if out.returncode != 0 or not out.stdout.strip():
    raise SystemExit('KEINE GPU aktiv. Laufzeit -> Laufzeittyp aendern -> GPU, dann neu starten.')
name, mem = [x.strip() for x in out.stdout.strip().split(',')]
mem_gb = int(mem.split()[0]) / 1024
print(f'GPU: {name} ({mem_gb:.1f} GB)')

# Die Batchgroesse muss zur Karte passen, sonst bricht das Training mit OOM ab.
BATCH_SIZE = 1 if mem_gb < 20 else 2
print(f'-> batch_size={BATCH_SIZE}')
print('Hinweis: T4 (16 GB) ist der haeufigste Gratis-Fall und reicht fuer batch_size=1.')


## 1. Trainingscode holen (MIT-lizenziert)


In [ ]:
%cd /content
![ -d Music-Source-Separation-Training ] || git clone --depth 1 https://github.com/ZFTurbo/Music-Source-Separation-Training.git
%cd /content/Music-Source-Separation-Training
!pip install -q -r requirements.txt 2>&1 | tail -5
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())


## 2. Datensatz

Zwei Wege. **Weg B ist der einzige, der zu einem kommerziell nutzbaren Modell führt.**

### Weg A — MUSDB18-HQ (Forschung/Entwicklung)
Braucht einen Zenodo-Account und die Zustimmung zu den Nutzungsbedingungen. Der Download-Link
ist personengebunden; trage ihn unten ein. ~30 GB, passt nicht auf die Colab-Systemplatte,
deshalb auf Google Drive ablegen.

### Weg B — eigene Multitracks
Lege deine eigenen Stems in dieser Struktur auf Drive ab. Das ist der Weg, wenn das Modell
später im Produkt landen soll:

```
dataset/
  train/
    Song A/  vocals.wav  drums.wav  bass.wav  other.wav
    Song B/  vocals.wav  drums.wav  bass.wav  other.wav
  valid/
    Song C/  vocals.wav  drums.wav  bass.wav  other.wav
```

Alle Dateien 44.1 kHz stereo WAV, pro Song gleich lang. Die Mischung wird beim Training
als Summe der Stems gebildet — genau wie dein `goldStandard.ts` es auch macht.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ---- HIER ANPASSEN ----------------------------------------------------------
DATA_ROOT  = '/content/drive/MyDrive/stem-training/dataset'   # enthaelt train/ und valid/
OUT_ROOT   = '/content/drive/MyDrive/stem-training/runs'      # Checkpoints (Drive = ueberlebt Abbruch)
# -----------------------------------------------------------------------------

import os
os.makedirs(OUT_ROOT, exist_ok=True)
train_dir, valid_dir = f'{DATA_ROOT}/train', f'{DATA_ROOT}/valid'

if not os.path.isdir(train_dir):
    raise SystemExit(f'Kein Trainingsordner unter {train_dir} — Abschnitt 2 lesen.')

songs = [d for d in sorted(os.listdir(train_dir)) if os.path.isdir(f'{train_dir}/{d}')]
print(f'{len(songs)} Trainings-Songs gefunden')

REQUIRED = ['vocals.wav','drums.wav','bass.wav','other.wav']
bad = [s for s in songs if not all(os.path.exists(f'{train_dir}/{s}/{f}') for f in REQUIRED)]
if bad:
    raise SystemExit(f'Diesen Songs fehlen Stems {REQUIRED}: {bad[:5]}')
if len(songs) < 10:
    print(f'WARNUNG: nur {len(songs)} Songs. Das reicht fuer einen Rauchtest,')
    print('aber nicht fuer ein Modell, das auf fremder Musik funktioniert.')
print('Datensatzstruktur ok.')


## 3. Startgewichte laden

Finetuning braucht einen Ausgangspunkt. Ohne `--start_check_point` würdest du doch bei Null
anfangen — und damit in Colab scheitern.


In [ ]:
import os, urllib.request
CKPT_DIR = '/content/pretrained'; os.makedirs(CKPT_DIR, exist_ok=True)

BASE_CKPT = f'{CKPT_DIR}/model_bs_roformer_ep_317_sdr_12.9755.ckpt'
BASE_CFG  = '/content/Music-Source-Separation-Training/configs/viperx/model_bs_roformer_ep_317_sdr_12.9755.yaml'
CKPT_URL  = ('https://github.com/TRvlvr/model_repo/releases/download/'
             'all_public_uvr_models/model_bs_roformer_ep_317_sdr_12.9755.ckpt')

if not os.path.exists(BASE_CKPT):
    print('lade Startgewichte (~600 MB) ...')
    urllib.request.urlretrieve(CKPT_URL, BASE_CKPT)
print('Checkpoint:', round(os.path.getsize(BASE_CKPT)/1e6), 'MB')

import hashlib
h = hashlib.sha256()
with open(BASE_CKPT,'rb') as f:
    for chunk in iter(lambda: f.read(1<<20), b''): h.update(chunk)
print('sha256:', h.hexdigest())
print('\nLIZENZ: fuer diese Gewichte ist keine explizite Lizenz veroeffentlicht.')
print('Alles, was daraus finetunt wird, ist bis zur Klaerung dev-only.')


## 4. Config anpassen

Zwei Eingriffe: Batchgröße an die GPU anpassen und die Laufzeit so begrenzen, dass regelmäßig
ein Checkpoint auf Drive landet. Colab-Sessions sterben unangekündigt — ein Training ohne
Zwischenstände ist verlorene Rechenzeit.


In [ ]:
import yaml, shutil
CFG_PATH = '/content/config_finetune.yaml'
shutil.copy(BASE_CFG, CFG_PATH)
cfg = yaml.safe_load(open(CFG_PATH))

cfg['training']['batch_size'] = BATCH_SIZE
cfg['training']['num_steps']  = 200     # Schritte pro 'Epoche' -> haeufige Checkpoints
cfg['training']['num_epochs'] = 200
cfg['training']['lr']         = 1.0e-05 # niedriger als beim Training von Null: wir feintunen
cfg['training']['use_amp']    = True

with open(CFG_PATH,'w') as f: yaml.safe_dump(cfg, f, sort_keys=False)

print('instruments   :', cfg['training'].get('instruments'))
print('target        :', cfg['training'].get('target_instrument'))
print('batch_size    :', cfg['training']['batch_size'])
print('lr            :', cfg['training']['lr'])
print('chunk_size    :', cfg['audio']['chunk_size'])


> **Zur Stem-Anzahl:** Der obige Checkpoint ist ein **2-Stem-Modell** (vocals/other).
> Deine App erwartet standardmäßig 4 Stems. Entweder du finetunst mehrere 2-Stem-Modelle,
> oder du startest von einem 4-Stem-Checkpoint mit passender Config. Lass die Instrumentenliste
> in der Config **unverändert** — sie muss zu den Startgewichten passen, sonst passt die
> Ausgabeschicht nicht und das Laden schlägt fehl.


## 5. Training

Läuft, bis die Session endet oder du stoppst. Jeder verbesserte Checkpoint wird nach Drive
geschrieben, du kannst also jederzeit abbrechen und später fortsetzen.


In [ ]:
!python train.py \
  --model_type bs_roformer \
  --config_path {CFG_PATH} \
  --start_check_point {BASE_CKPT} \
  --results_path {OUT_ROOT} \
  --data_path {train_dir} \
  --valid_path {valid_dir} \
  --num_workers 2 \
  --device_ids 0


### Fortsetzen nach Session-Abbruch
Zelle unten statt der obigen ausführen — sie nimmt den besten Checkpoint aus Drive als Start.


In [ ]:
import glob, os
cands = sorted(glob.glob(f'{OUT_ROOT}/*.ckpt'), key=os.path.getmtime)
if not cands:
    print('Kein Checkpoint in Drive — mit Zelle 5 neu starten.')
else:
    RESUME = cands[-1]
    print('setze fort bei:', RESUME)
    !python train.py --model_type bs_roformer --config_path {CFG_PATH} \
      --start_check_point {RESUME} --results_path {OUT_ROOT} \
      --data_path {train_dir} --valid_path {valid_dir} --num_workers 2 --device_ids 0


## 6. Ehrliche Bewertung

Die Trainings-Loss sagt **nichts** darüber, ob das Modell in deiner App taugt. Hier wird
gegen Songs geprüft, die das Modell nie gesehen hat. Ein SDR, das auf `valid` gut aussieht,
aber auf fremder Musik einbricht, ist Überanpassung — bei kleinen Datensätzen der Normalfall.


In [ ]:
import glob, os
best = sorted(glob.glob(f'{OUT_ROOT}/*.ckpt'), key=os.path.getmtime)[-1]
print('bewerte:', best)
!python valid.py --model_type bs_roformer --config_path {CFG_PATH} \
  --start_check_point {best} --valid_path {valid_dir} --device_ids 0


**Einordnung der SDR-Werte** (vocals, Multisong-Benchmark): Referenzmodelle liegen bei 10–12 dB.
Unter ~6 dB ist das Ergebnis für ein Produkt nicht brauchbar. Vergleiche immer mit dem
**Startcheckpoint** — wenn dein Finetune schlechter ist als der Ausgangspunkt, hat das Training
geschadet (zu hohe LR, zu wenig/zu einseitige Daten).


## 7. Export in die App


In [ ]:
import shutil, os, hashlib, json, glob
EXPORT = '/content/drive/MyDrive/stem-training/export'
os.makedirs(EXPORT, exist_ok=True)

best = sorted(glob.glob(f'{OUT_ROOT}/*.ckpt'), key=os.path.getmtime)[-1]
MODEL_NAME = 'airdox_bs_roformer_finetune_v1'
ckpt_out = f'{EXPORT}/{MODEL_NAME}.ckpt'
cfg_out  = f'{EXPORT}/{MODEL_NAME}.yaml'
shutil.copy(best, ckpt_out); shutil.copy(CFG_PATH, cfg_out)

def sha256(p):
    h = hashlib.sha256()
    with open(p,'rb') as f:
        for c in iter(lambda: f.read(1<<20), b''): h.update(c)
    return h.hexdigest()

digest = sha256(ckpt_out)
meta = {
    'modelName': MODEL_NAME,
    'sha256': digest,
    'baseCheckpoint': os.path.basename(BASE_CKPT),
    'trainedOn': os.path.basename(DATA_ROOT),
    'licenseStatus': 'DEV_ONLY — Startgewichte ohne explizite Lizenz; Daten ggf. nicht-kommerziell',
}
json.dump(meta, open(f'{EXPORT}/{MODEL_NAME}.json','w'), indent=2)

print('exportiert nach', EXPORT)
print('sha256:', digest)
print()
print('Lokal installieren:')
print(f'  mkdir -p models/bsroformer')
print(f'  cp {MODEL_NAME}.ckpt {MODEL_NAME}.yaml models/bsroformer/')
print(f'  export STEM_MODEL_FILENAME={MODEL_NAME}.ckpt')
print()
print('Dann in der App pruefen lassen:')
print('  npm run test:stems      # Gate faellt das Qualitaetsurteil')


## 8. Gegenprobe in der App — der einzige Test, der zählt

Lade Checkpoint + Config auf deinen Rechner und lass **dein Gate** urteilen:

```bash
mkdir -p models/bsroformer
cp airdox_bs_roformer_finetune_v1.{ckpt,yaml} models/bsroformer/
export STEM_MODEL_FILENAME=airdox_bs_roformer_finetune_v1.ckpt
npm run test:stems
```

Das Gate kennt genau drei Urteile:

| Ergebnis | Bedeutung |
| --- | --- |
| `TECHNICAL_FAIL` | Pipeline kaputt — Stems fehlen, Original verändert, Absturz |
| `TECHNICAL_PASS_QUALITY_FAIL` | Läuft sauber, trennt aber zu schlecht |
| `RELEASE_READY` | technisch **und** qualitativ ausreichend |

Solange dort nicht `RELEASE_READY` steht, ist das Modell nicht fertig — unabhängig davon,
wie gut die Loss-Kurve aussah. Genau dafür hast du das Gate gebaut.
